# lipshift, Step 2: batch feature extraction

Runtime: T4 GPU. Safe to rerun in a new session: finished clips are skipped.

Secrets (key icon, Notebook access ON): `HF_TOKEN` (Hugging Face, read) and `GH_TOKEN`
(GitHub fine-grained token for `lipshift`, Contents + Workflows read/write). Pass secret
names to `userdata.get`, never the values.

Google Drive layout expected under `MyDrive/lipshift/`:
- `weights/vsr_trlrs2lrs3vox2avsp_base.pth` (from Step 0)
- `weights/multilingual/LRS3_V_WER32.3.zip` and `CMUMOSEAS_V_ES_WER44.5.zip` (from Step 0b)
- `patches/*.patch`: code updates to apply and push (optional; already-applied ones are skipped)

Outputs: `MyDrive/lipshift/features/<encoder>/*.npy`, `features/log.jsonl`, `features/manifest.csv`.
Run Cell 1, then Runtime > Restart session, then Cells 2 onward.

In [ ]:
# Cell 1: installs and upstream code. Then Runtime > Restart session.
!pip install -q mediapipe==1.2.2 sentencepiece scikit-image six pytest==9.2.0
!git clone -q https://github.com/mpc001/auto_avsr /content/auto_avsr 2>/dev/null || true
!git -C /content/auto_avsr checkout -q 182b62837773ab01052d4ac21ef1d2203ea7d267
!git clone -q https://github.com/mpc001/Visual_Speech_Recognition_for_Multiple_Languages /content/vsrml 2>/dev/null || true
!git -C /content/vsrml checkout -q 5e1405db0ae816509fb312f9a578724c2e0de0c7
!wget -q -nc https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite -O /content/blaze_face_short_range.tflite
!ls -l /content/blaze_face_short_range.tflite
print("Done. Now: Runtime > Restart session, then run Cell 2.")

In [ ]:
# Cell 2: clone your repo, apply new patches (tests must pass), push, install
GITHUB_USER = "Elia0911"
GIT_NAME = "Eli OUN"
GIT_EMAIL = ""        # your GitHub no-reply address, same as in Step 1
assert GIT_EMAIL, "Fill GIT_EMAIL"
import os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive")
WORK = Path("/content/drive/MyDrive/lipshift")
REPO = Path("/content/lipshift")
HELPER = "!f() { echo username=x-access-token; echo password=$LIPSHIFT_GH_TOKEN; }; f"
def git(*args, check=True, auth=False):
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    pre = []
    if auth:
        env["LIPSHIFT_GH_TOKEN"] = userdata.get("GH_TOKEN")
        pre = ["-c", f"credential.helper={HELPER}"]
    r = subprocess.run(["git", *pre, *args], cwd=REPO if REPO.exists() else "/content",
                       capture_output=True, text=True, env=env)
    out = (r.stdout + r.stderr).strip()
    if auth:
        out = out.replace(env["LIPSHIFT_GH_TOKEN"], "***")
    if check and r.returncode != 0:
        raise RuntimeError(out)
    return out
if not REPO.exists():
    git("clone", "-q", f"https://github.com/{GITHUB_USER}/lipshift.git", str(REPO), auth=True)
git("config", "user.name", GIT_NAME); git("config", "user.email", GIT_EMAIL)
git("pull", "-q", "--ff-only", auth=True)
applied = set(git("log", "--format=%s").splitlines())
new = []
import email, re
for p in sorted((WORK / "patches").glob("*.patch")):
    # Header parser handles folded lines; strip "[PATCH n/m]" to get the commit subject
    raw = str(email.message_from_string(p.read_text())["Subject"] or "")
    subject = re.sub(r"^\[PATCH[^\]]*\]\s*", "", " ".join(raw.split()))
    if not subject:
        raise RuntimeError(f"No Subject header in {p.name}")
    if subject not in applied:
        git("am", "-q", "--committer-date-is-author-date", str(p))
        new.append(subject)
print("applied:", new or "nothing new")
!pip install -q -e /content/lipshift --no-deps
if new:
    # make you the author of the applied commits, then test before pushing
    git("rebase", "-q", "--exec", "git commit -q --amend --no-edit --reset-author", "origin/main")
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "-m", "not integration"],
                       cwd=REPO, capture_output=True, text=True)
    print(r.stdout[-600:])
    assert r.returncode == 0, "Tests failed: not pushing. Send me the output above."
    print(git("push", "-q", "origin", "main", auth=True) or "pushed")
print(git("log", "--oneline", "-5"))
print(subprocess.run([sys.executable, "-c", "import lipshift.extract.batch as b; print('batch module OK', b.__file__)"],
                     cwd="/tmp", capture_output=True, text=True).stdout)

In [ ]:
# Cell 3: MAVOS-DD metadata and the full counts table
import json
import numpy as np, pandas as pd
from huggingface_hub import snapshot_download
from lipshift.data.subset import load_mavos_metadata, cell_counts
HF_TOKEN = userdata.get("HF_TOKEN")
REPO_ID = "unibuc-cs/MAVOS-DD"
META_DIR = Path("/content/mavos_meta")
snapshot_download(REPO_ID, repo_type="dataset", local_dir=META_DIR, token=HF_TOKEN,
                  ignore_patterns=["*.mp4", "*.MP4", "*.wav"])
df = load_mavos_metadata(META_DIR)
FEAT = WORK / "features"; FEAT.mkdir(parents=True, exist_ok=True)
cell_counts(df).to_csv(WORK / "step0" / "mavos_counts_full.csv", index=False)
pd.set_option("display.width", 200)
print(df.shape)
print(pd.crosstab([df["language"]], [df["label"], df["generative_method"]]).to_string())
print(df.groupby(["split", "open_set_model", "open_set_language"]).size().to_string())

In [ ]:
# Cell 4: stratified subset (the manifest is frozen on first run so every session uses the same clips)
from lipshift.data.subset import select_subset
PER_CELL = 30            # clips per (language, generator, label, split) cell
LANGUAGES = None         # None = all languages
SEED = 0
SEC_PER_CLIP_EST = 5.0   # ASSUMPTION from Step 0 (3 clips); replaced by the measured value after Cell 6
params = {"per_cell": PER_CELL, "languages": LANGUAGES, "seed": SEED}
MANIFEST, PARAMS = FEAT / "manifest.csv", FEAT / "manifest_params.json"
if MANIFEST.exists():
    old = json.loads(PARAMS.read_text())
    assert old == params, f"Manifest exists with {old}. Keep those values, or delete features/manifest.* to start over."
    manifest = pd.read_csv(MANIFEST)
else:
    sub = df if LANGUAGES is None else df[df["language"].isin(LANGUAGES)]
    manifest = select_subset(sub, per_cell=PER_CELL, seed=SEED)
    manifest.to_csv(MANIFEST, index=False); PARAMS.write_text(json.dumps(params))
n_enes = int(manifest["language"].isin(["english", "spanish"]).sum())
print(f"manifest: {len(manifest)} clips ({n_enes} English/Spanish, used by all 3 encoders)")
print(f"rough GPU time: {len(manifest) * SEC_PER_CLIP_EST / 3600:.1f} h (estimate, not measured)")
print(f"rough Drive space: {(len(manifest) * 0.3 + n_enes * 0.2) / 1024:.1f} GB (float16 features, ~10 s clips)")
print(cell_counts(manifest).groupby("language")["n"].sum().to_string())

In [ ]:
# Cell 5: load the three encoders (weights copied from Drive to local disk first)
import hashlib, shutil, zipfile, torch
from lipshift.extract.encoders import AutoAVSREncoder, MultilingualVSREncoder
from lipshift.extract.batch import EncoderSpec
AUTO_AVSR, VSRML = "/content/auto_avsr", "/content/vsrml"
FACE_MODEL = "/content/blaze_face_short_range.tflite"
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"
LOCALW = Path("/content/weights"); LOCALW.mkdir(exist_ok=True)
ck = LOCALW / "vsr_trlrs2lrs3vox2avsp_base.pth"
if not ck.exists():
    shutil.copyfile(WORK / "weights" / ck.name, ck)
assert hashlib.md5(ck.read_bytes()).hexdigest().startswith("49f77"), "Auto-AVSR checkpoint MD5 mismatch"
confs = {}
for z in ["LRS3_V_WER32.3.zip", "CMUMOSEAS_V_ES_WER44.5.zip"]:
    lz = LOCALW / z
    if not lz.exists():
        shutil.copyfile(WORK / "weights" / "multilingual" / z, lz)
    zf = zipfile.ZipFile(lz); assert zf.testzip() is None, f"{z} corrupted"
    dst = LOCALW / lz.stem
    if not list(dst.rglob("model.pth")):
        zf.extractall(dst)
    confs[z] = next(dst.rglob("model.json"))
enc_core = AutoAVSREncoder(AUTO_AVSR)
enc_ml_en = MultilingualVSREncoder(VSRML, model_conf=confs["LRS3_V_WER32.3.zip"])
enc_ml_es = MultilingualVSREncoder(VSRML, model_conf=confs["CMUMOSEAS_V_ES_WER44.5.zip"])
for name, e, path in [("core", enc_core, ck),
                      ("ml_en", enc_ml_en, confs["LRS3_V_WER32.3.zip"].parent / "model.pth"),
                      ("ml_es", enc_ml_es, confs["CMUMOSEAS_V_ES_WER44.5.zip"].parent / "model.pth")]:
    k = e.load_checkpoint(path)
    assert not k["missing"] and not k["unexpected"], f"{name}: {len(k['missing'])} missing, {len(k['unexpected'])} unexpected"
    print(name, "loaded")
ENES = frozenset({"english", "spanish"})
ENCODERS = [EncoderSpec("autoavsr_en", enc_core),
            EncoderSpec("ml_en", enc_ml_en, ENES),
            EncoderSpec("ml_es", enc_ml_es, ENES)]

In [ ]:
# Cell 6: run until the time budget, then stop cleanly (rerun later to continue)
import time
from huggingface_hub import hf_hub_download
from lipshift.extract.batch import run_batch, summarize_progress
from lipshift.extract.pipeline import prepare_clip
MAX_MINUTES = 150        # stop starting new clips after this; keep below your session limit
CLIPS = Path("/content/clips"); CLIPS.mkdir(exist_ok=True)
def resolve(vp):
    return hf_hub_download(REPO_ID, filename=vp, repo_type="dataset", local_dir=CLIPS, token=HF_TOKEN)
def cleanup(path):
    Path(path).unlink(missing_ok=True)
rows = manifest[["video_path", "language"]].to_dict("records")
t_start = time.time()
counts = run_batch(rows, resolve=resolve,
                   prepare=lambda p: prepare_clip(p, AUTO_AVSR, FACE_MODEL, max_frames=300),
                   encoders=ENCODERS, out_root=FEAT, device="cuda", cleanup=cleanup,
                   max_minutes=MAX_MINUTES, max_errors=200, prefetch=4)
print(counts)

In [ ]:
# Cell 7: progress, measured speed and errors
log = pd.read_json(FEAT / "log.jsonl", lines=True)
ok = log[log["status"] == "ok"]
progress = summarize_progress(rows, ENCODERS, FEAT)
print(json.dumps(progress, indent=2))
remaining = sum(v["applicable"] - v["done"] for k, v in progress.items() if k == "autoavsr_en")
if len(ok):
    sec = float(ok["sec_total"].median())
    print(f"measured median {sec:.1f} s per clip over {len(ok)} clips -> ~{remaining * sec / 3600:.1f} GPU h left for core")
err = log[log["status"] == "error"]
if len(err):
    print(f"{len(err)} failed attempts; most common:")
    print(err["error"].str.slice(0, 90).value_counts().head(8).to_string())
(WORK / "step0" / "extract_progress.json").write_text(json.dumps({"progress": progress, "counts_last_run": counts}, indent=2))